# Strong Focusing

### How Alternating Gradients Made the Modern Particle Accelerator Possible

By 1950 accelerator builders had run into a wall. Every synchrotron used **weak
focusing**: the bending magnet was shaped so its field fell off gently with radius,
which gives a mild restoring force in both transverse directions at once. It works,
but the restoring force is so weak that the beam wanders over a large area. The
Cosmotron at Brookhaven, then the world's most energetic proton machine at 3 GeV,
needed a vacuum chamber roughly 15 cm tall and 60 cm wide, wrapped in about 2,000
tons of steel. Doubling the energy meant roughly doubling the ring, and the magnet
cross section had to stay just as large. Cost grew faster than energy, and the next
generation of machines looked unaffordable.

**Strong focusing** broke that scaling. The idea is almost perverse: instead of a
magnet that focuses weakly in both planes, use a magnet that focuses *hard* in one
plane and *defocuses just as hard* in the other, then alternate. The net effect of
the sequence is strong focusing in **both** planes. Beam sizes fell from tens of
centimeters to a few centimeters, magnet apertures shrank with them, and the steel
bill collapsed. Every large circular accelerator since - the CERN Proton
Synchrotron, Brookhaven's AGS, the Tevatron, the LHC - is built on this one idea.

### Who thought of it

**Nicholas Christofilos** got there first. A Greek-American engineer who ran an
elevator-maintenance business in Athens, he taught himself accelerator theory from
journals and in 1949-1950 worked out alternating-gradient focusing on his own. He
filed a U.S. patent rather than publishing in a journal, and mailed his ideas to
Berkeley, where they were set aside. Almost nobody in the field knew.

In the summer of 1952 the idea was rediscovered independently at Brookhaven.
**M. Stanley Livingston** wondered whether alternating the orientation of the
Cosmotron's C-shaped magnets might cancel a saturation effect. **Ernest Courant**
did the calculation expecting a small correction and instead found something that
looked wrong: the *stronger* he made the alternating gradients, the *better* the
beam was focused. **Hartland Snyder** recognized the optical analogy - a converging
lens followed by a diverging lens of equal strength is net converging - and with
**John Blewett** the group had the theory within days. When Christofilos learned of
the Brookhaven paper he produced his patent, and the group publicly acknowledged his
priority.

The idea was theoretical first and it spread fast. Cornell had an alternating-
gradient electron synchrotron running by 1954. CERN scrapped its planned 10 GeV
weak-focusing machine and built the 25 GeV Proton Synchrotron instead, for about the
same projected cost. Brookhaven's Alternating Gradient Synchrotron reached 33 GeV in
1960.

### What this notebook does

We follow the design of a small proton synchrotron from first principles, in the
same order a real designer works:

1. Look inside a single quadrupole and see why it focuses one plane and defocuses
   the other.
2. Show, by ray tracing, why alternating them nonetheless confines the beam.
3. Build the transfer matrices for drifts, dipoles, and quadrupoles.
4. Lay out a FODO lattice and check that it is stable.
5. Choose the quadrupole strengths to land on a safe working point (the tune).
6. Compute the Twiss parameters, the beta functions, and the beam envelope.
7. Watch the phase-space ellipse turn without changing area.
8. Track protons, then break the machine with a single steering error.
9. Replace the exact element maps with symplectic integrators, and see why
   accelerator physicists invented them in the first place.

Everything uses only NumPy, SciPy, and Matplotlib - no accelerator package.

---
## 1. Inside a quadrupole

A quadrupole magnet has four poles arranged around the beam, adjacent poles of
opposite polarity. Near the axis its field is

$$B_x = G\,y, \qquad B_y = G\,x,$$

where $G = \partial B_y / \partial x$ is the **field gradient**, measured in tesla
per meter. Notice what this says: the field is exactly **zero on the axis** and
grows **linearly** with distance from it. A quadrupole has no useful "field
strength" of its own - only a gradient.

The pole faces are machined to the shape of the magnetic equipotentials. Since
$\mathbf{B} = -\nabla \phi_m$ with $\phi_m = -G\,x\,y$, the equipotentials are the
hyperbolas $xy = \pm r_0^2/2$, where $r_0$ is the bore radius. That is why the pole
tips of a real quadrupole are hyperbolic and sit at $45^\circ$ to the axes.

### The force on a proton

A proton moving along the beam direction $\hat{z}$ with speed $v$ feels the Lorentz
force $\mathbf{F} = q\,\mathbf{v} \times \mathbf{B}$. With
$\mathbf{v} = v\hat{z}$ and $\mathbf{B} = (Gy,\ Gx,\ 0)$,

$$\mathbf{F} = qv\,(-B_y,\ B_x,\ 0) = qvG\,(-x,\ +y,\ 0).$$

The horizontal force is $-qvGx$: always back toward the axis, and proportional to
displacement. That is exactly the force law of a lens. The vertical force is
$+qvGy$: always **away** from the axis. One magnet cannot do both.

### Focusing strength

Dividing the force by the momentum gives the trajectory equation
$x'' + k\,x = 0$ with

$$k = \frac{qG}{p} = \frac{G}{B\rho}, \qquad B\rho\,[\text{T m}] \approx 3.3356\;p\,[\text{GeV}/c].$$

$B\rho$ is the **magnetic rigidity** - how hard the particle is to bend. A stiffer
(higher momentum) beam needs a stronger gradient for the same focusing.

In [ ]:
"""strong_focusing.ipynb"""

# Cell 01 - Imports and the beam we are going to accelerate

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Wedge
from matplotlib.ticker import NullFormatter
from scipy.optimize import fsolve

BRHO_PER_GEV = 3.3356  # T m of rigidity per GeV/c, for a singly charged particle
PROTON_MASS = 0.93827  # GeV/c^2

MOMENTUM = 3.0  # GeV/c
BRHO = BRHO_PER_GEV * MOMENTUM

total_energy = np.hypot(MOMENTUM, PROTON_MASS)
kinetic_energy = total_energy - PROTON_MASS
speed_ratio = MOMENTUM / total_energy

print(f"proton momentum   p     = {MOMENTUM:.3f} GeV/c")
print(f"total energy      E     = {total_energy:.4f} GeV")
print(f"kinetic energy    T     = {kinetic_energy:.4f} GeV")
print(f"speed             v / c = {speed_ratio:.4f}")
print(f"magnetic rigidity B*rho = {BRHO:.4f} T m")
print()
print(f"a 10 T/m gradient gives k = {10.0 / BRHO:.4f} 1/m^2  (expected about 1.0)")

In [ ]:
# Cell 02 - Plot 1: the quadrupole field and the force it puts on a proton

BORE_RADIUS = 0.030  # 30 mm from the axis to the pole tip
GRADIENT = 13.2  # T/m, the value we will end up designing for


def quadrupole_field(x, y, gradient=GRADIENT):
    """Return (Bx, By) of an ideal quadrupole at position (x, y)."""
    return gradient * y, gradient * x


span = 0.045
axis = np.linspace(-span, span, 400)
grid_x, grid_y = np.meshgrid(axis, axis)
field_x, field_y = quadrupole_field(grid_x, grid_y)
field_magnitude = np.hypot(field_x, field_y)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Left: field magnitude, field lines, and the hyperbolic pole tips
ax = axes[0]
surf = ax.contourf(grid_x, grid_y, field_magnitude, levels=24, cmap="rainbow")
fig.colorbar(surf, ax=ax, shrink=0.8, label="|B| (T)")
ax.streamplot(
    grid_x,
    grid_y,
    field_x,
    field_y,
    color="k",
    density=1.1,
    linewidth=0.6,
    arrowsize=0.8,
)

# Pole faces follow the equipotentials x*y = +/- r0^2 / 2
u = np.linspace(0.006, 0.075, 200)
half_product = BORE_RADIUS**2 / 2.0
for sign_x, sign_y, pole in [(1, 1, "N"), (-1, -1, "N"), (1, -1, "S"), (-1, 1, "S")]:
    pole_x = sign_x * u
    pole_y = sign_y * half_product / u
    inside = np.abs(pole_y) < span * 1.6
    ax.plot(pole_x[inside], pole_y[inside], color="k", lw=4)
    ax.text(
        sign_x * 0.030,
        sign_y * 0.030,
        pole,
        ha="center",
        va="center",
        fontsize=13,
        fontweight="bold",
        bbox={"boxstyle": "circle", "fc": "white", "ec": "k"},
    )
ax.plot(0, 0, "wo", ms=8, mec="k")
ax.set_xlim(-span, span)
ax.set_ylim(-span, span)
ax.set_aspect("equal")
ax.set_title("Field lines and hyperbolic pole tips")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")

# Right: F = q v x B for a proton traveling into the page
ax = axes[1]
sample = np.linspace(-0.028, 0.028, 9)
sample_x, sample_y = np.meshgrid(sample, sample)
sample_bx, sample_by = quadrupole_field(sample_x, sample_y)
ax.quiver(sample_x, sample_y, -sample_by, sample_bx, color="crimson", pivot="mid")
ax.axhline(0, color="k", lw=0.8)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-span, span)
ax.set_ylim(-span, span)
ax.set_aspect("equal")
ax.set_title("Lorentz force on a proton moving into the page")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")

fig.suptitle("A Single Quadrupole Focuses Horizontally and Defocuses Vertically")
fig.tight_layout()
plt.show()

print(f"|B| at the pole tip radius = {GRADIENT * BORE_RADIUS:.4f} T")
print(f"k for this gradient        = {GRADIENT / BRHO:.4f} 1/m^2")

The force arrows point **inward** along the horizontal axis and **outward** along
the vertical axis. That asymmetry is not a design flaw and cannot be engineered
away. In the field-free bore Maxwell requires $\nabla \cdot \mathbf{B} = 0$ and
$\nabla \times \mathbf{B} = 0$, so the two focusing strengths must be equal and
opposite: $k_x = -k_y$ always. **A magnetic lens that focuses both planes at once
does not exist.**

---
## 2. Why alternating gradients still work

Here is the piece that surprised everyone in 1952. Put a focusing lens of focal
length $f$ and a defocusing lens of focal length $-f$ a distance $L$ apart. Multiply
the thin-lens matrices and read off the combined focal length:

$$\frac{1}{f_\text{net}} = \frac{L}{f^2} > 0.$$

The pair is **net focusing**, whichever lens comes first, and the result does not
depend on the sign. The physical reason is that the two lenses do not act on the
same beam. A particle enters the focusing lens far from the axis, gets a big inward
kick, and therefore arrives at the defocusing lens *closer* to the axis, where the
outward kick is weaker. Focusing wins because a quadrupole kick is proportional to
displacement, and the displacement has changed in between.

This also explains Courant's "impossible" result. Making the gradients stronger
makes both kicks bigger, but it also makes the displacement difference bigger, and
the net focusing goes as $1/f^2$. Stronger alternating gradients focus better, right
up to the stability limit we will find in section 5.

The next plot traces protons leaving a point on the axis at different angles. The
dotted gray lines are the same particles with the magnets switched off.

In [ ]:
# Cell 03 - Plot 2: ray tracing through a string of alternating thin lenses

FOCAL_LENGTH = 1.70  # m
LENS_SPACING = 2.40  # m
LENS_COUNT = 11
CHAMBER = 0.025  # vacuum chamber half aperture, 25 mm


def trace_alternating(angle0, focal, spacing, n_lenses, n_sub=40):
    """Trace one ray from the axis through alternating thin lenses.

    Parameters
    ----------
    angle0 : float
        Initial divergence angle in radians.
    focal : float
        Focal length of the first lens; the next lens uses -focal, and so on.
    spacing : float
        Drift length between lenses.
    n_lenses : int
        How many lenses to pass through.
    n_sub : int
        Steps used to draw each drift, for a smooth line.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        Distance along the beamline and transverse position.
    """
    distance = [0.0]
    position = [0.0]
    x, angle, s = 0.0, angle0, 0.0
    for i in range(n_lenses):
        this_focal = focal if i % 2 == 0 else -focal
        angle -= x / this_focal  # the thin-lens kick
        for _ in range(n_sub):
            x += angle * spacing / n_sub
            s += spacing / n_sub
            distance.append(s)
            position.append(x)
    return np.array(distance), np.array(position)


# 1/f_net = L / f^2 for a focus-defocus pair, checked against the matrix product
lens_f = np.array([[1.0, 0.0], [-1.0 / FOCAL_LENGTH, 1.0]])
lens_d = np.array([[1.0, 0.0], [+1.0 / FOCAL_LENGTH, 1.0]])
drift = np.array([[1.0, LENS_SPACING], [0.0, 1.0]])
doublet = lens_d @ drift @ lens_f
print(f"focus-defocus doublet matrix:\n{np.round(doublet, 5)}")
print(f"net focal length from the matrix: {-1.0 / doublet[1, 0]:.5f} m")
print(
    f"net focal length from L / f^2:    "
    f"{FOCAL_LENGTH**2 / LENS_SPACING:.5f} m  (they agree)"
)

angles = np.linspace(-0.0025, 0.0025, 9)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for ax, sign, plane, coord in zip(
    axes, [1, -1], ["Horizontal", "Vertical"], ["x", "y"], strict=True
):
    for angle0 in angles:
        s_arr, x_arr = trace_alternating(
            angle0, sign * FOCAL_LENGTH, LENS_SPACING, LENS_COUNT
        )
        ax.plot(s_arr, x_arr * 1e3, lw=1.0)
        ax.plot(s_arr, angle0 * s_arr * 1e3, color="gray", ls=":", lw=0.7)
    for i in range(LENS_COUNT):
        focuses_here = (i % 2 == 0) == (sign > 0)
        ax.axvline(
            i * LENS_SPACING,
            lw=7,
            alpha=0.30,
            color="royalblue" if focuses_here else "crimson",
        )
    ax.axhline(CHAMBER * 1e3, color="k", ls="--", lw=1.2)
    ax.axhline(-CHAMBER * 1e3, color="k", ls="--", lw=1.2)
    ax.set_ylim(-40, 40)
    ax.set_ylabel(f"{coord} (mm)")
    ax.set_title(f"{plane} plane")
    ax.grid(True)

legend_items = [
    Line2D([], [], color="royalblue", lw=7, alpha=0.30, label="focusing lens"),
    Line2D([], [], color="crimson", lw=7, alpha=0.30, label="defocusing lens"),
    Line2D([], [], color="gray", ls=":", lw=1.0, label="same ray, magnets off"),
    Line2D([], [], color="k", ls="--", lw=1.2, label="chamber wall"),
]
axes[0].legend(handles=legend_items, fontsize=8, loc="upper right", ncols=2)
axes[1].set_xlabel("distance along the beamline (m)")
fig.suptitle("Alternating Gradients Confine the Beam in Both Planes")
fig.tight_layout()
plt.show()

Every ray stays bounded, in **both** planes, and every ray keeps coming back to the
axis. With the magnets off, the same particles leave the chamber within about six
meters. The two planes are not identical - the plane that starts at a focusing lens
swings wider - but neither one runs away.

That is the whole idea. Everything below is bookkeeping: how to describe it
precisely enough to design a machine with it.

---
## 3. Transfer matrices

For small deviations from the ideal orbit, a particle is described by

$$\mathbf{u} = \begin{bmatrix} x \\ x' \end{bmatrix},
\qquad x' = \frac{dx}{ds},$$

where $s$ is distance along the beamline, $x$ is displacement from the ideal orbit,
and $x'$ is the angle to it. Each element multiplies this vector by a $2 \times 2$
matrix, and elements compose by matrix multiplication - with the **first** element
on the **right**:

$$\mathbf{u}_\text{out} = M_n \cdots M_2 M_1 \mathbf{u}_\text{in}.$$

Solving $x'' + kx = 0$ over a length $L$ gives all three matrices at once:

$$M_\text{drift} = \begin{bmatrix} 1 & L \\ 0 & 1 \end{bmatrix}, \qquad
M_{k>0} = \begin{bmatrix} \cos\sqrt{k}L & \frac{1}{\sqrt{k}}\sin\sqrt{k}L \\
-\sqrt{k}\sin\sqrt{k}L & \cos\sqrt{k}L \end{bmatrix}, \qquad
M_{k<0} = \begin{bmatrix} \cosh\sqrt{|k|}L & \frac{1}{\sqrt{|k|}}\sinh\sqrt{|k|}L \\
\sqrt{|k|}\sinh\sqrt{|k|}L & \cosh\sqrt{|k|}L \end{bmatrix}.$$

The same quadrupole uses $k$ in one plane and $-k$ in the other, so one function
covers focusing, defocusing, and (at $k = 0$) drifting.

A bending dipole is not just a drift. A particle at larger radius travels through a
slightly longer path in the field and gets bent back toward the ideal orbit, which
is a weak horizontal focusing with $k = 1/\rho^2$ and nothing in the vertical plane.
This is exactly the **weak focusing** that pre-1952 machines relied on entirely, and
we will see how feeble it is compared to a quadrupole.

Every one of these matrices has determinant 1. That is not a coincidence - it is
Liouville's theorem in matrix form, and it is what will keep the emittance constant
in section 8.

In [ ]:
# Cell 04 - The transfer matrix for any element


def transfer_matrix(length: float, k: float) -> np.ndarray:
    """Return the 2x2 transfer matrix for a length of beamline.

    Parameters
    ----------
    length : float
        Length of the element in meters.
    k : float
        Focusing strength in 1/m^2. Positive focuses, negative defocuses,
        zero is a field-free drift.

    Returns
    -------
    np.ndarray
        The 2x2 matrix acting on the column vector [x, x'].
    """
    if abs(k) < 1e-12:
        return np.array([[1.0, length], [0.0, 1.0]])

    if k > 0.0:
        root_k = np.sqrt(k)
        phase = root_k * length
        return np.array(
            [
                [np.cos(phase), np.sin(phase) / root_k],
                [-root_k * np.sin(phase), np.cos(phase)],
            ]
        )

    root_k = np.sqrt(-k)
    phase = root_k * length
    return np.array(
        [
            [np.cosh(phase), np.sinh(phase) / root_k],
            [root_k * np.sinh(phase), np.cosh(phase)],
        ]
    )


# Quick checks: the focal length of a real quadrupole is -1 / M[1, 0], which
# approaches the thin-lens value 1 / (k L) only as the magnet gets shorter
print("focal length of a quadrupole of strength k = 1.3 1/m^2")
print(f"{'length (m)':>12}{'exact (m)':>12}{'thin lens (m)':>15}{'error':>9}")
for test_length in [0.40, 0.20, 0.10, 0.05]:
    test_matrix = transfer_matrix(test_length, 1.3)
    exact = -1.0 / test_matrix[1, 0]
    thin = 1.0 / (1.3 * test_length)
    print(f"{test_length:12.2f}{exact:12.4f}{thin:15.4f}{exact / thin - 1:8.1%}")

print("\nthe matrix for our quadrupoles (k = 1.3 1/m^2, L = 0.4 m):")
print(np.round(transfer_matrix(0.4, 1.3), 5))
print(
    f"determinant of that matrix = "
    f"{np.linalg.det(transfer_matrix(0.4, 1.3)):.12f}  (expected 1)"
)
print(
    f"determinant of a drift     = "
    f"{np.linalg.det(transfer_matrix(2.0, 0.0)):.12f}  (expected 1)"
)

---
## 4. Step one: lay out the ring

Now we design a machine. The choices at this stage are geometric, and they are
driven by how much bending we need and how much room the magnets take.

We want a small proton synchrotron holding a 3 GeV/c beam. The ring must bend the
beam through $2\pi$ in total, so if we use $N_B$ identical dipoles each bends
$\theta = 2\pi / N_B$ and has bending radius $\rho = L_B / \theta$. The field
follows from the rigidity, $B = B\rho / \rho$.

The repeating unit is a **FODO cell**, the simplest useful lattice:

```
QF/2   drift   dipole   drift   QD   drift   dipole   drift   QF/2
```

"FODO" is focus - O (nothing) - defocus - O. Starting and ending on a half QF makes
the cell symmetric about its center, which forces $\alpha = 0$ at both ends and
makes the periodic solution in section 6 particularly clean. Two dipoles are tucked
into the drift spaces because bending and focusing are both needed and space is
expensive.

In [ ]:
# Cell 05 - Lattice geometry

N_CELLS = 20  # FODO cells around the ring
L_QUAD = 0.40  # full quadrupole length (m)
L_BEND = 1.50  # dipole length (m)
L_DRIFT = 0.25  # drift between magnets (m)

CELL_LENGTH = 2 * L_QUAD + 4 * L_DRIFT + 2 * L_BEND
CIRCUMFERENCE = N_CELLS * CELL_LENGTH

N_DIPOLES = 2 * N_CELLS
BEND_ANGLE = 2 * np.pi / N_DIPOLES
BEND_RADIUS = L_BEND / BEND_ANGLE
DIPOLE_FIELD = BRHO / BEND_RADIUS
K_BEND = 1.0 / BEND_RADIUS**2  # the dipole's weak horizontal focusing


def build_cell(k_focus: float, k_defocus: float) -> list[tuple]:
    """Return one FODO cell as a list of (name, kind, length, kx, ky).

    Parameters
    ----------
    k_focus : float
        Strength of the QF quadrupoles, which focus horizontally.
    k_defocus : float
        Strength of the QD quadrupoles, which focus vertically.

    Returns
    -------
    list[tuple]
        One entry per element, with its focusing strength in each plane.
    """
    return [
        ("QF", "quad", L_QUAD / 2, +k_focus, -k_focus),
        ("D", "drift", L_DRIFT, 0.0, 0.0),
        ("B", "dipole", L_BEND, K_BEND, 0.0),
        ("D", "drift", L_DRIFT, 0.0, 0.0),
        ("QD", "quad", L_QUAD, -k_defocus, +k_defocus),
        ("D", "drift", L_DRIFT, 0.0, 0.0),
        ("B", "dipole", L_BEND, K_BEND, 0.0),
        ("D", "drift", L_DRIFT, 0.0, 0.0),
        ("QF", "quad", L_QUAD / 2, +k_focus, -k_focus),
    ]


print(f"cells around the ring   = {N_CELLS}")
print(f"cell length             = {CELL_LENGTH:.2f} m")
print(f"circumference           = {CIRCUMFERENCE:.2f} m")
print(f"dipoles                 = {N_DIPOLES}")
print(f"bend angle per dipole   = {np.degrees(BEND_ANGLE):.2f} deg")
print(f"bending radius          = {BEND_RADIUS:.4f} m")
print(f"dipole field            = {DIPOLE_FIELD:.4f} T")
print(f"dipole weak focusing k  = {K_BEND:.6f} 1/m^2")
print()
print(f"total bend = {N_DIPOLES * np.degrees(BEND_ANGLE):.1f} deg  (expected 360)")
print(f"cell adds up to {sum(e[2] for e in build_cell(1.0, 1.0)):.2f} m")

In [ ]:
# Cell 06 - Plot 3: the magnet lattice, one cell and the whole ring

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: a linear layout of a single cell
ax = axes[0]
position = 0.0
for name, kind, length, kx, _ in build_cell(1.0, 1.0):
    if kind == "quad":
        color = "royalblue" if kx > 0 else "crimson"
        height = 0.6 if kx > 0 else -0.6
        ax.add_patch(Rectangle((position, 0), length, height, fc=color, ec="k"))
        ax.text(
            position + length / 2,
            height + np.sign(height) * 0.18,
            name,
            ha="center",
            va="center",
            fontsize=9,
        )
    elif kind == "dipole":
        ax.add_patch(
            Rectangle((position, -0.25), length, 0.5, fc="darkseagreen", ec="k")
        )
        ax.text(position + length / 2, 0, "B", ha="center", va="center", fontsize=9)
    position += length
ax.axhline(0, color="k", lw=1)
ax.annotate(
    "beam",
    xy=(CELL_LENGTH, 0.85),
    xytext=(CELL_LENGTH - 1.3, 0.85),
    arrowprops={"arrowstyle": "->", "lw": 1.5},
    va="center",
    fontsize=9,
)
ax.set_xlim(-0.3, CELL_LENGTH + 0.3)
ax.set_ylim(-1.3, 1.3)
ax.set_yticks([])
ax.set_xlabel("s (m)")
ax.set_title(f"One FODO cell ({CELL_LENGTH:.1f} m)")

# Right: the same cell repeated around the ring (schematic, drawn as a circle)
ax = axes[1]
ring_radius = CIRCUMFERENCE / (2 * np.pi)
angle = np.linspace(0, 2 * np.pi, 400)
ax.plot(
    ring_radius * np.cos(angle), ring_radius * np.sin(angle), color="lightgray", lw=1
)
position = 0.0
for _ in range(N_CELLS):
    for _name, kind, length, kx, _ky in build_cell(1.0, 1.0):
        start = np.degrees(2 * np.pi * position / CIRCUMFERENCE)
        stop = np.degrees(2 * np.pi * (position + length) / CIRCUMFERENCE)
        if kind == "dipole":
            ax.add_patch(
                Wedge(
                    (0, 0),
                    ring_radius + 0.5,
                    start,
                    stop,
                    width=1.0,
                    fc="darkseagreen",
                    ec="none",
                )
            )
        elif kind == "quad":
            ax.add_patch(
                Wedge(
                    (0, 0),
                    ring_radius + 1.1,
                    start,
                    stop,
                    width=2.2,
                    fc="royalblue" if kx > 0 else "crimson",
                    ec="none",
                )
            )
        position += length
ax.set_xlim(-ring_radius * 1.25, ring_radius * 1.25)
ax.set_ylim(-ring_radius * 1.25, ring_radius * 1.25)
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_title(f"{N_CELLS} cells, {N_DIPOLES} dipoles, {CIRCUMFERENCE:.0f} m around")
ax.legend(
    handles=[
        Rectangle((0, 0), 1, 1, fc="royalblue", label="QF (focuses x)"),
        Rectangle((0, 0), 1, 1, fc="crimson", label="QD (focuses y)"),
        Rectangle((0, 0), 1, 1, fc="darkseagreen", label="dipole (bends)"),
    ],
    fontsize=8,
    loc="center",
)

fig.suptitle("The Magnet Lattice")
fig.tight_layout()
plt.show()

---
## 5. Step two: stability, phase advance, and the tune

Multiply the matrices of one cell together and you get the **one-cell matrix**
$M$. A particle that goes around $n$ cells is described by $M^n$, so the whole
question of whether the machine works reduces to: do the powers of $M$ stay bounded?

Because $\det M = 1$, the eigenvalues are $e^{\pm i\mu}$ and

$$\cos\mu = \frac{\operatorname{Tr}(M)}{2}.$$

If $\left|\operatorname{Tr}(M)/2\right| < 1$ then $\mu$ is real, the eigenvalues sit
on the unit circle, and the motion is a bounded oscillation. If it exceeds 1 then
$\mu$ becomes imaginary, one eigenvalue is larger than 1, and the amplitude grows
exponentially - the beam is lost within a few turns. **That single inequality is the
stability criterion for the entire machine**, and it has to hold in both planes.

The angle $\mu$ is the **phase advance per cell**. Summed over $N$ cells it gives
the **tune**,

$$Q = \frac{N\mu}{2\pi},$$

the number of transverse oscillations a particle makes per revolution.

### Picking the quadrupole strengths

A thin-lens estimate gets us close. For a FODO cell of length $L_\text{cell}$ with
lenses of focal length $f$,

$$\sin\frac{\mu}{2} = \frac{L_\text{cell}}{4f},$$

so aiming for $\mu = 90^\circ$ (a common choice, near the minimum of the beam size
for a given aperture) fixes $f$, and $k \approx 1/(fL_q)$.

That estimate ignores the finite length of the quadrupole and the weak focusing of
the dipoles, so we then solve for the exact answer. Because $Q$ near an integer or a
simple fraction is dangerous - errors then add up in phase turn after turn, a
**resonance** - we do not accept whatever tune falls out. We power the QF and QD
magnets from two separate supplies and solve numerically for the pair
$(k_F, k_D)$ that puts $(Q_x, Q_y)$ exactly where we want it. This is called
**tune matching**, and it is what MAD-X does for a living.

In [ ]:
# Cell 07 - Phase advance, tune, and matching the working point


def cell_matrices(k_focus: float, k_defocus: float) -> tuple[np.ndarray, np.ndarray]:
    """Return the one-cell transfer matrices in the (x, y) planes."""
    matrix_x = np.eye(2)
    matrix_y = np.eye(2)
    for _name, _kind, length, kx, ky in build_cell(k_focus, k_defocus):
        matrix_x = transfer_matrix(length, kx) @ matrix_x
        matrix_y = transfer_matrix(length, ky) @ matrix_y
    return matrix_x, matrix_y


def phase_advance(matrix: np.ndarray) -> float:
    """Return the phase advance mu of a stable one-cell matrix, else NaN."""
    half_trace = 0.5 * np.trace(matrix)
    if abs(half_trace) >= 1.0:
        return np.nan
    mu = np.arccos(half_trace)
    # arccos only covers 0 to pi; the sign of M12 tells us which half we are in
    return 2 * np.pi - mu if matrix[0, 1] < 0 else mu


def ring_tunes(k_focus: float, k_defocus: float) -> tuple[float, float]:
    """Return the horizontal and vertical tunes of the whole ring."""
    matrix_x, matrix_y = cell_matrices(k_focus, k_defocus)
    return (
        N_CELLS * phase_advance(matrix_x) / (2 * np.pi),
        N_CELLS * phase_advance(matrix_y) / (2 * np.pi),
    )


# The thin-lens starting guess for 90 degrees of phase advance per cell
focal_guess = (CELL_LENGTH / 2) / (2 * np.sin(np.radians(90.0) / 2))
k_guess = 1.0 / (focal_guess * L_QUAD)
guess_x, guess_y = ring_tunes(k_guess, k_guess)
matrix_x_guess, _ = cell_matrices(k_guess, k_guess)

print("thin-lens estimate for 90 degrees per cell")
print(f"  focal length f = {focal_guess:.4f} m")
print(f"  k              = {k_guess:.4f} 1/m^2  (G = {k_guess * BRHO:.3f} T/m)")
print(f"  exact mu_x     = {np.degrees(phase_advance(matrix_x_guess)):.2f} deg")
print(f"  exact Qx, Qy   = {guess_x:.4f}, {guess_y:.4f}")

# Now match the two families to a working point chosen off the resonance lines
TARGET_QX, TARGET_QY = 4.28, 4.19


def tune_error(strengths: np.ndarray) -> list[float]:
    """Difference between the tunes at these strengths and the target tunes."""
    tune_x, tune_y = ring_tunes(strengths[0], strengths[1])
    if np.isnan(tune_x) or np.isnan(tune_y):
        return [10.0, 10.0]  # steer the solver away from unstable settings
    return [tune_x - TARGET_QX, tune_y - TARGET_QY]


# fsolve hands back an array of the two strengths, so pull them out explicitly
matched = np.asarray(fsolve(tune_error, [1.3, 1.3]), dtype=float)
K_FOCUS = float(matched[0])
K_DEFOCUS = float(matched[1])

matrix_x_cell, matrix_y_cell = cell_matrices(K_FOCUS, K_DEFOCUS)
mu_x = phase_advance(matrix_x_cell)
mu_y = phase_advance(matrix_y_cell)
tune_x, tune_y = ring_tunes(K_FOCUS, K_DEFOCUS)

print("\nmatched working point")
print(f"  k_F = {K_FOCUS:.4f} 1/m^2   gradient = {K_FOCUS * BRHO:6.3f} T/m")
print(f"  k_D = {K_DEFOCUS:.4f} 1/m^2   gradient = {K_DEFOCUS * BRHO:6.3f} T/m")
print(
    f"  phase advance per cell: {np.degrees(mu_x):.3f} deg (x), "
    f"{np.degrees(mu_y):.3f} deg (y)"
)
print(f"  Qx = {tune_x:.4f}  (target {TARGET_QX})")
print(f"  Qy = {tune_y:.4f}  (target {TARGET_QY})")
print(f"  Tr(Mx) / 2 = {0.5 * np.trace(matrix_x_cell):+.5f}")
print(f"  Tr(My) / 2 = {0.5 * np.trace(matrix_y_cell):+.5f}   (both must be < 1)")
print(
    f"\nquadrupole focusing is {K_FOCUS / K_BEND:.0f} times stronger than the "
    f"weak focusing of the dipoles"
)

In [ ]:
# Cell 08 - Plot 4: the stability band and the tune diagram

k_scan = np.linspace(0.02, 2.4, 700)
half_trace_x = np.array([0.5 * np.trace(cell_matrices(k, k)[0]) for k in k_scan])
half_trace_y = np.array([0.5 * np.trace(cell_matrices(k, k)[1]) for k in k_scan])
stable = (np.abs(half_trace_x) < 1) & (np.abs(half_trace_y) < 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax = axes[0]
ax.axhspan(-1, 1, color="palegreen", alpha=0.5, label="stable band")
ax.plot(
    k_scan, half_trace_x, color="royalblue", label=r"horizontal $\mathrm{Tr}(M_x)/2$"
)
ax.plot(k_scan, half_trace_y, color="crimson", label=r"vertical $\mathrm{Tr}(M_y)/2$")
ax.axvline(K_FOCUS, color="k", ls="--", lw=1, label=f"our $k$ = {K_FOCUS:.3f}")
ax.set_ylim(-3, 2)
ax.set_xlabel(r"quadrupole strength $k$ (m$^{-2}$)")
ax.set_ylabel(r"$\mathrm{Tr}(M)/2$")
ax.legend(fontsize=8, loc="lower left")
ax.grid(True)
ax.set_title(r"A cell is stable only where $|\mathrm{Tr}(M)/2| < 1$")

ax = axes[1]
low, high = 3.95, 4.65
line_style = {1: ("black", 1.6), 2: ("royalblue", 1.0), 3: ("seagreen", 0.7)}
for m in range(-3, 4):
    for n in range(-3, 4):
        order = abs(m) + abs(n)
        if order == 0 or order > 3:
            continue
        color, width = line_style[order]
        for p in range(-40, 41):
            if n != 0:
                edge = np.array([low, high])
                ax.plot(edge, (p - m * edge) / n, color=color, lw=width, alpha=0.7)
            else:
                ax.axvline(p / m, color=color, lw=width, alpha=0.7)
ax.plot(tune_x, tune_y, "r*", ms=20, zorder=5)
ax.annotate(
    f"working point\n({tune_x:.2f}, {tune_y:.2f})",
    xy=(tune_x, tune_y),
    xytext=(tune_x + 0.03, tune_y - 0.10),
    fontsize=9,
    zorder=5,
)
ax.set_xlim(low, high)
ax.set_ylim(low, high)
ax.set_xlabel(r"$Q_x$")
ax.set_ylabel(r"$Q_y$")
ax.legend(
    handles=[
        Line2D([], [], color=c, lw=w, label=f"order {o}")
        for o, (c, w) in line_style.items()
    ],
    fontsize=8,
    loc="upper right",
    framealpha=1.0,
    facecolor="white",
)
ax.set_title("Tune diagram, resonance lines to third order")

fig.suptitle("Choosing the Quadrupole Strengths")
fig.tight_layout()
plt.show()

print(
    f"stable for k between {k_scan[stable].min():.3f} and "
    f"{k_scan[stable].max():.3f} 1/m^2"
)
print(f"our choice k_F = {K_FOCUS:.3f} sits comfortably inside that band")

The left panel is the whole reason strong focusing has a limit. Increase $k$ and
the phase advance per cell rises; push past the edge of the green band and the cell
stops being a lens at all. Courant's "stronger is better" holds only up to there.

The right panel is the tune diagram, and every line on it is a value of
$(Q_x, Q_y)$ to stay away from. A line $mQ_x + nQ_y = p$ with integers $m, n, p$ is
a resonance of order $|m| + |n|$: whenever the tunes satisfy it, a small magnet
imperfection kicks the beam at the same phase every turn and the error accumulates
instead of averaging away. Integer lines are the most dangerous (a steering error
drives them, as we will see in section 9), then half integers (a gradient error),
then third order. Real machines are operated in the gaps.

---
## 6. Step three: Twiss parameters and the beta function

Tracking every proton would be wasteful. Since the motion is linear, the whole beam
can be described by an ellipse in the $(x, x')$ phase plane, and the ellipse by
three numbers - the **Twiss** or **Courant-Snyder** parameters $\beta$, $\alpha$,
$\gamma$:

$$\gamma x^2 + 2\alpha x x' + \beta x'^2 = \varepsilon,
\qquad \beta\gamma - \alpha^2 = 1.$$

- $\beta(s)$ sets the width of the beam. The rms size is $\sigma = \sqrt{\varepsilon\beta}$.
- $\alpha(s) = -\tfrac{1}{2}\,d\beta/ds$ says whether the beam is converging
  ($\alpha > 0$), diverging ($\alpha < 0$), or at a waist ($\alpha = 0$).
- $\gamma(s)$ sets the spread of angles.
- $\varepsilon$ is the **emittance**, the area of the ellipse divided by $\pi$.

These have nothing to do with the relativistic $\beta$ and $\gamma$. They are purely
geometric.

For a ring the solution has to repeat every cell. Writing the one-cell matrix in
Twiss form,

$$M = \begin{bmatrix} \cos\mu + \alpha\sin\mu & \beta\sin\mu \\
-\gamma\sin\mu & \cos\mu - \alpha\sin\mu \end{bmatrix},$$

and reading off the entries gives the **periodic** solution

$$\beta = \frac{M_{12}}{\sin\mu}, \qquad
\alpha = \frac{M_{11} - M_{22}}{2\sin\mu}, \qquad
\gamma = \frac{1 + \alpha^2}{\beta}.$$

There is no freedom left: the lattice decides its own beta function. To get
$\beta(s)$ everywhere we slice the ring into short pieces and push the parameters
through each one with

$$\begin{bmatrix} \beta_2 \\ \alpha_2 \\ \gamma_2 \end{bmatrix} =
\begin{bmatrix} a^2 & -2ab & b^2 \\ -ac & ad + bc & -bd \\ c^2 & -2cd & d^2 \end{bmatrix}
\begin{bmatrix} \beta_1 \\ \alpha_1 \\ \gamma_1 \end{bmatrix},
\qquad M = \begin{bmatrix} a & b \\ c & d \end{bmatrix}.$$

In [ ]:
# Cell 09 - Periodic Twiss parameters and how to propagate them

SLICE_LENGTH = 0.01  # 1 cm steps, fine enough to draw smooth curves


def periodic_twiss(matrix: np.ndarray) -> tuple[float, float, float]:
    """Return (beta, alpha, gamma) that repeat under this one-cell matrix."""
    mu = phase_advance(matrix)
    if np.isnan(mu):
        raise ValueError("the cell is unstable, so no periodic solution exists")
    sin_mu = np.sin(mu)
    beta = matrix[0, 1] / sin_mu
    alpha = (matrix[0, 0] - matrix[1, 1]) / (2.0 * sin_mu)
    return beta, alpha, (1.0 + alpha**2) / beta


def slice_lattice(k_focus: float, k_defocus: float, n_cells: int):
    """Chop n_cells of lattice into short slices.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        Position at the end of each slice, the horizontal and vertical
        focusing strength of each slice, and each slice's length.
    """
    positions, k_x, k_y = [], [], []
    position = 0.0
    for _ in range(n_cells):
        for _name, _kind, length, kx, ky in build_cell(k_focus, k_defocus):
            n_slices = max(1, round(length / SLICE_LENGTH))
            for _ in range(n_slices):
                position += length / n_slices
                positions.append(position)
                k_x.append(kx)
                k_y.append(ky)
    positions = np.array(positions)
    lengths = np.diff(np.concatenate(([0.0], positions)))
    return positions, np.array(k_x), np.array(k_y), lengths


def propagate_twiss(beta0, alpha0, k_slices, length_slices):
    """Push (beta, alpha) through every slice and record the result."""
    beta, alpha = beta0, alpha0
    gamma = (1.0 + alpha0**2) / beta0
    beta_out = np.empty(len(k_slices) + 1)
    alpha_out = np.empty(len(k_slices) + 1)
    beta_out[0], alpha_out[0] = beta, alpha
    for i, (k, length) in enumerate(zip(k_slices, length_slices, strict=True)):
        a, b, c, d = transfer_matrix(length, k).ravel()
        beta, alpha, gamma = (
            a * a * beta - 2 * a * b * alpha + b * b * gamma,
            -a * c * beta + (a * d + b * c) * alpha - b * d * gamma,
            c * c * beta - 2 * c * d * alpha + d * d * gamma,
        )
        beta_out[i + 1], alpha_out[i + 1] = beta, alpha
    return beta_out, alpha_out


BETA_X0, ALPHA_X0, GAMMA_X0 = periodic_twiss(matrix_x_cell)
BETA_Y0, ALPHA_Y0, GAMMA_Y0 = periodic_twiss(matrix_y_cell)

print("periodic Twiss parameters at the center of a QF")
print(f"  beta_x  = {BETA_X0:8.4f} m     beta_y  = {BETA_Y0:8.4f} m")
print(f"  alpha_x = {ALPHA_X0:8.1e}       alpha_y = {ALPHA_Y0:8.1e}")
print(f"  gamma_x = {GAMMA_X0:8.4f} 1/m   gamma_y = {GAMMA_Y0:8.4f} 1/m")
print(f"  beta*gamma - alpha^2 = {BETA_X0 * GAMMA_X0 - ALPHA_X0**2:.12f} (expected 1)")
print("\nalpha is zero because the cell is symmetric about the QF center,")
print("so that point is a waist in both planes.")

N_PLOT_CELLS = 3
s_slice, kx_slice, ky_slice, ds_slice = slice_lattice(K_FOCUS, K_DEFOCUS, N_PLOT_CELLS)
s_plot = np.concatenate(([0.0], s_slice))
beta_x, alpha_x = propagate_twiss(BETA_X0, ALPHA_X0, kx_slice, ds_slice)
beta_y, alpha_y = propagate_twiss(BETA_Y0, ALPHA_Y0, ky_slice, ds_slice)

print(f"\nbeta_x runs from {beta_x.min():.3f} to {beta_x.max():.3f} m")
print(f"beta_y runs from {beta_y.min():.3f} to {beta_y.max():.3f} m")
print(
    f"after {N_PLOT_CELLS} cells beta_x has changed by "
    f"{beta_x[-1] - beta_x[0]:.2e} m, so the solution really is periodic"
)

In [ ]:
# Cell 10 - Plot 5: the beta functions through three cells


def draw_lattice_ribbon(ax, n_cells, bottom, height):
    """Draw a strip of magnet symbols underneath a plot of s."""
    position = 0.0
    for _ in range(n_cells):
        for _name, kind, length, kx, _ky in build_cell(K_FOCUS, K_DEFOCUS):
            if kind == "quad":
                ax.add_patch(
                    Rectangle(
                        (position, bottom),
                        length,
                        height,
                        fc="royalblue" if kx > 0 else "crimson",
                        ec="none",
                    )
                )
            elif kind == "dipole":
                ax.add_patch(
                    Rectangle(
                        (position, bottom + height * 0.25),
                        length,
                        height * 0.5,
                        fc="darkseagreen",
                        ec="none",
                    )
                )
            position += length


fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(s_plot, beta_x, color="royalblue", lw=1.8, label=r"$\beta_x(s)$")
ax.plot(s_plot, beta_y, color="crimson", lw=1.8, label=r"$\beta_y(s)$")
draw_lattice_ribbon(ax, N_PLOT_CELLS, -1.5, 1.0)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlim(-0.2, s_plot[-1] + 0.2)
ax.set_ylim(-1.9, 9.5)
ax.set_xlabel("s (m)")
ax.set_ylabel(r"$\beta$ (m)")
ax.legend(loc="upper right")
ax.grid(True)
ax.set_title("Beta Functions Through Three FODO Cells")
fig.tight_layout()
plt.show()

The two curves are the same shape shifted by half a cell. $\beta_x$ peaks in the QF
and dips in the QD; $\beta_y$ does the opposite. That is the alternating gradient
written in the language of beam optics: the beam is wide where it is being focused
horizontally and narrow where it is being defocused, which is precisely why the
focusing wins.

Note the scale. $\beta$ swings between roughly 2 m and 8 m, a factor of four in
$\beta$ and a factor of two in beam size, and it does this every 4.8 m forever. In a
weak-focusing machine $\beta$ would be comparable to the ring radius, about 15 m
here, and the beam would be correspondingly larger everywhere.

---
## 7. Step four: emittance and the beam envelope

The beta function says how the beam size varies; the **emittance** says how big it
is. Emittance is a property of the beam, not the lattice - it is fixed when the beam
is injected and (Liouville) does not change afterwards. Together they give the rms
size at every point,

$$\sigma_x(s) = \sqrt{\varepsilon_x\,\beta_x(s)}, \qquad
\sigma_y(s) = \sqrt{\varepsilon_y\,\beta_y(s)}.$$

Designing the vacuum chamber means making sure that a beam several sigma wide fits
with room to spare everywhere in the ring, including at the peaks of $\beta$. This
is the plot that decides how big, and therefore how expensive, the magnets are.

In [ ]:
# Cell 11 - Plot 6: the beam envelope inside the vacuum chamber

EMITTANCE_X = 5.0e-6  # m rad, a typical injected geometric emittance
EMITTANCE_Y = 5.0e-6
HALF_APERTURE = 0.025  # 25 mm chamber half height

sigma_x = np.sqrt(EMITTANCE_X * beta_x)
sigma_y = np.sqrt(EMITTANCE_Y * beta_y)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, sigma, color, coord in zip(
    axes, [sigma_x, sigma_y], ["royalblue", "crimson"], ["x", "y"], strict=True
):
    ax.fill_between(
        s_plot,
        -3 * sigma * 1e3,
        3 * sigma * 1e3,
        color=color,
        alpha=0.22,
        label=r"$\pm 3\sigma$",
    )
    ax.plot(s_plot, sigma * 1e3, color=color, lw=1.6, label=r"$\pm \sigma$")
    ax.plot(s_plot, -sigma * 1e3, color=color, lw=1.6)
    ax.axhline(HALF_APERTURE * 1e3, color="k", ls="--", lw=1.2, label="vacuum chamber")
    ax.axhline(-HALF_APERTURE * 1e3, color="k", ls="--", lw=1.2)
    draw_lattice_ribbon(ax, N_PLOT_CELLS, -29, 3)
    ax.set_ylabel(f"{coord} (mm)")
    ax.set_ylim(-31, 31)
    ax.legend(fontsize=8, loc="upper right", ncols=3)
    ax.grid(True)
axes[1].set_xlabel("s (m)")
fig.suptitle("Beam Envelope Inside the Vacuum Chamber")
fig.tight_layout()
plt.show()

print(
    f"largest sigma_x = {sigma_x.max() * 1e3:.3f} mm (at beta_x = {beta_x.max():.2f} m)"
)
print(
    f"largest sigma_y = {sigma_y.max() * 1e3:.3f} mm (at beta_y = {beta_y.max():.2f} m)"
)
print(
    f"a 3 sigma beam fills {3 * sigma_x.max() / HALF_APERTURE * 100:.0f}% "
    f"of the half aperture"
)
print(
    f"the beam would need emittance "
    f"{(HALF_APERTURE / 3) ** 2 / beta_x.max() * 1e6:.2f} mm mrad to touch "
    f"the wall at 3 sigma"
)

---
## 8. Step five: phase space and Liouville's theorem

Now watch the same beam in phase space. We generate a Gaussian bunch matched to the
lattice - meaning its covariance matrix is

$$\Sigma = \varepsilon \begin{bmatrix} \beta & -\alpha \\ -\alpha & \gamma \end{bmatrix}$$

- and push it to three places in the cell with the transfer matrix.

Both the $1\sigma$ and $2\sigma$ ellipses are drawn. A real bunch is Gaussian, not
a hard-edged ellipse, so only about 39% of the particles fall inside the $1\sigma$
ellipse and 86% inside the $2\sigma$ one - the ellipse marks the shape of the
distribution, not its boundary.

The ellipse changes shape constantly. In the QF it is wide and flat: large spread in
position, small spread in angle. Half a cell later in the QD it is tall and narrow:
the beam has been squeezed in position and paid for it in angle. In between it is
tilted, which is what a nonzero $\alpha$ means.

What never changes is the **area**. Every transfer matrix has determinant 1, so the
map is area preserving, which is Liouville's theorem for linear optics. The
emittance we measure from the particle distribution is
$\varepsilon = \sqrt{\det \Sigma}$, and it comes out the same at all three places to
machine precision.

This is the sharpest statement of what magnets can and cannot do. Quadrupoles
**reshape** the beam. They never improve it. A small emittance has to be created at
the source, or removed later by a genuinely dissipative process such as synchrotron
radiation damping or stochastic cooling - both of which break the assumptions above.

In [ ]:
# Cell 12 - Plot 7: the phase-space ellipse at three points in the cell


def matrix_to(s_target: float, plane: str = "x") -> np.ndarray:
    """Return the transfer matrix from the cell start to a distance s_target."""
    matrix = np.eye(2)
    position = 0.0
    for _name, _kind, length, kx, ky in build_cell(K_FOCUS, K_DEFOCUS):
        remaining = s_target - position
        if remaining <= 1e-9:
            break
        step = min(length, remaining)
        matrix = transfer_matrix(step, kx if plane == "x" else ky) @ matrix
        position += step
    return matrix


rng = np.random.default_rng(2024)
sigma_matrix = EMITTANCE_X * np.array([[BETA_X0, -ALPHA_X0], [-ALPHA_X0, GAMMA_X0]])
bunch = rng.multivariate_normal([0.0, 0.0], sigma_matrix, size=3000).T
print(f"generated {bunch.shape[1]} protons matched to the lattice")

locations = [0.0, CELL_LENGTH / 4, CELL_LENGTH / 2]
labels = ["QF center (s = 0)", "middle of the first dipole", "QD center"]
angle = np.linspace(0, 2 * np.pi, 300)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, s_target, label in zip(axes, locations, labels, strict=True):
    matrix = matrix_to(s_target)
    moved = matrix @ bunch
    a, b = matrix[0, 0], matrix[0, 1]
    c, d = matrix[1, 0], matrix[1, 1]
    beta_here = a * a * BETA_X0 - 2 * a * b * ALPHA_X0 + b * b * GAMMA_X0
    alpha_here = -a * c * BETA_X0 + (a * d + b * c) * ALPHA_X0 - b * d * GAMMA_X0
    measured = np.sqrt(np.linalg.det(np.cov(moved)))
    ax.scatter(moved[0] * 1e3, moved[1] * 1e3, s=1, alpha=0.25, color="navy")
    for n_sigma, style in [(1, "-"), (2, "--")]:
        ellipse_x = n_sigma * np.sqrt(EMITTANCE_X * beta_here) * np.cos(angle)
        ellipse_xp = (
            -n_sigma
            * np.sqrt(EMITTANCE_X / beta_here)
            * (alpha_here * np.cos(angle) + np.sin(angle))
        )
        ax.plot(
            ellipse_x * 1e3,
            ellipse_xp * 1e3,
            color="orange",
            lw=2,
            ls=style,
            label=rf"{n_sigma}$\sigma$ ellipse",
        )
    ax.set_title(
        f"{label}\n"
        rf"$\beta_x$ = {beta_here:.2f} m,  "
        rf"$\alpha_x$ = {alpha_here:+.2f},  "
        rf"$\varepsilon$ = {measured * 1e6:.3f} mm mrad"
    )
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("x' (mrad)")
    ax.set_xlim(-20, 20)
    ax.set_ylim(-4, 4)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True)

fig.suptitle("The Phase-Space Ellipse Turns and Shears, but its Area Never Changes")
fig.tight_layout()
plt.show()

for s_target, label in zip(locations, labels, strict=True):
    moved = matrix_to(s_target) @ bunch
    print(
        f"emittance at {label:28s} = "
        f"{np.sqrt(np.linalg.det(np.cov(moved))) * 1e6:.6f} mm mrad"
    )
print(
    f"determinant of every transfer matrix = "
    f"{np.linalg.det(matrix_to(CELL_LENGTH / 2)):.12f}"
)

---
## 9. Step six: betatron oscillation, steering, and why the tune matters

A single proton does not travel down the middle of the pipe. It oscillates about
the ideal orbit, and that oscillation is called a **betatron oscillation** - named
after the betatron, the machine where it was first analyzed. Its solution is

$$x(s) = \sqrt{2J\,\beta(s)}\;\cos\big(\varphi(s) - \varphi_0\big),
\qquad \varphi(s) = \int_0^s \frac{ds'}{\beta(s')},$$

an oscillation whose amplitude is modulated by $\sqrt{\beta(s)}$ and whose local
wavelength is set by $\beta$ as well. The number of oscillations per turn is the
tune $Q$.

The first two panels below track one proton for one turn. The fast zigzag is the
cell structure - twenty quadrupole kicks per turn - and the slow swing underneath is
the betatron oscillation itself, about four and a quarter of them, exactly $Q$.

### Steering

Dipoles are what steer the beam. That is a virtue when it is deliberate and a
problem when it is not: a single magnet that is slightly misaligned or slightly
mispowered acts as an unintended steering kick $\theta$, and the beam settles onto a
new **closed orbit** - the one trajectory that reproduces itself after a full turn.
Solving $\mathbf{u} = M(\mathbf{u} + \boldsymbol{\theta})$ gives it directly:

$$\mathbf{u}_\text{co} = (I - M)^{-1} M \boldsymbol{\theta}.$$

The distortion does not stay near the offending magnet; it rings all the way around
the ring, with the same $Q$ oscillations per turn. Its size scales as

$$x_\text{co}(s) \propto \frac{\theta\sqrt{\beta_k \beta(s)}}{2\sin \pi Q},$$

which blows up as $Q$ approaches an integer. That is the integer resonance made
concrete, and it is why the working point in section 5 was chosen so carefully. It
is also why alignment was one of the hardest engineering problems in the first
strong-focusing machines: the tolerances are fractions of a millimeter, on magnets
weighing tons, over hundreds of meters.

In [ ]:
# Cell 13 - Plot 8: tracking one proton, then adding a steering error

s_ring, kx_ring, ky_ring, ds_ring = slice_lattice(K_FOCUS, K_DEFOCUS, N_CELLS)
s_ring_plot = np.concatenate(([0.0], s_ring))
beta_x_ring, _ = propagate_twiss(BETA_X0, ALPHA_X0, kx_ring, ds_ring)
beta_y_ring, _ = propagate_twiss(BETA_Y0, ALPHA_Y0, ky_ring, ds_ring)


def track_particle(x0, angle0, k_slices, length_slices):
    """Track one particle slice by slice and return its position along s."""
    position = np.empty(len(k_slices) + 1)
    position[0] = x0
    state = np.array([x0, angle0])
    for i, (k, length) in enumerate(zip(k_slices, length_slices, strict=True)):
        state = transfer_matrix(length, k) @ state
        position[i + 1] = state[0]
    return position


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, beta_ring, k_ring, start, color, coord, tune in zip(
    [axes[0, 0], axes[0, 1]],
    [beta_x_ring, beta_y_ring],
    [kx_ring, ky_ring],
    [0.005, 0.0025],
    ["royalblue", "crimson"],
    ["x", "y"],
    [tune_x, tune_y],
    strict=True,
):
    trajectory = track_particle(start, 0.0, k_ring, ds_ring)
    action = start**2 / beta_ring[0]  # 2J, fixed by the starting condition
    envelope = np.sqrt(action * beta_ring)
    ax.fill_between(
        s_ring_plot,
        -envelope * 1e3,
        envelope * 1e3,
        color=color,
        alpha=0.18,
        label=r"$\pm\sqrt{2J\,\beta(s)}$",
    )
    ax.plot(s_ring_plot, trajectory * 1e3, color="k", lw=0.9, label="one proton")
    ax.set_title(f"{coord} betatron oscillation: {tune:.2f} per turn")
    ax.set_xlabel("s (m)")
    ax.set_ylabel(f"{coord} (mm)")
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(True)

# One steering error of 1 mrad, and the closed orbit it creates
STEERING_KICK = 1.0e-3  # rad
ring_matrix = np.linalg.matrix_power(matrix_x_cell, N_CELLS)
kick_vector = np.array([0.0, STEERING_KICK])
closed_orbit_start = np.linalg.solve(np.eye(2) - ring_matrix, ring_matrix @ kick_vector)
closed_orbit = track_particle(
    closed_orbit_start[0], closed_orbit_start[1] + STEERING_KICK, kx_ring, ds_ring
)

ax = axes[1, 0]
ax.plot(
    s_ring_plot,
    closed_orbit * 1e3,
    color="darkorange",
    lw=1.2,
    label="closed orbit after one 1 mrad kick",
)
ax.axhline(0, color="k", lw=1, ls="--", label="ideal orbit")
ax.plot(0, closed_orbit[0] * 1e3, "ko", ms=5)
ax.annotate(
    "steering error here",
    xy=(0, closed_orbit[0] * 1e3),
    xytext=(7, 5.8),
    arrowprops={"arrowstyle": "->"},
    fontsize=8,
)
ax.set_ylim(-7.5, 7.5)
ax.set_xlabel("s (m)")
ax.set_ylabel("x (mm)")
ax.set_title("One steering error distorts the orbit all the way around")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True)

ax = axes[1, 1]
tune_scan = np.linspace(4.03, 4.97, 800)
distortion = np.abs(STEERING_KICK * BETA_X0 / (2 * np.sin(np.pi * tune_scan))) * 1e3
ax.plot(tune_scan, distortion, color="purple")
ax.axhline(
    HALF_APERTURE * 1e3, color="k", ls="--", lw=1.2, label="chamber half aperture"
)
ax.plot(
    tune_x,
    abs(STEERING_KICK * BETA_X0 / (2 * np.sin(np.pi * tune_x))) * 1e3,
    "r*",
    ms=16,
    label=f"our $Q_x$ = {tune_x:.2f}",
)
ax.set_ylim(0, 40)
ax.set_xlabel(r"$Q_x$")
ax.set_ylabel("peak orbit distortion (mm)")
ax.set_title("The same error is amplified without limit near an integer tune")
ax.legend(fontsize=8, loc="upper center")
ax.grid(True)

fig.suptitle("Betatron Oscillation, Steering, and Why the Tune Matters")
fig.tight_layout()
plt.show()

print(
    f"closed orbit at the kick: x = {closed_orbit_start[0] * 1e3:.3f} mm, "
    f"x' = {closed_orbit_start[1] * 1e3:.3f} mrad"
)
print(
    f"it closes on itself to within "
    f"{abs(closed_orbit[-1] - closed_orbit[0]) * 1e9:.3f} nm after one turn"
)
print(
    f"peak distortion = {np.abs(closed_orbit).max() * 1e3:.3f} mm "
    f"from a single 1 mrad error"
)
print(
    f"that is {np.abs(closed_orbit).max() / sigma_x.max():.1f} times the largest "
    f"beam sigma - an orbit error the size of the beam itself, from one magnet"
)

A 1 mrad kick - roughly what a 1 mm misalignment of one quadrupole produces - moves
the orbit by about 5 mm, which is nearly the whole beam. Multiply by forty magnets
with independent errors and it is clear why the first alternating-gradient machines
needed a new generation of beam-position monitors and steering correctors before
they would run at all.

---
## 10. Symplectic tracking, and where Yoshida comes in

Nothing above needed a numerical integrator. Inside an element $k$ is constant, so
$x'' + kx = 0$ has a closed-form solution and the transfer matrix **is** the exact
answer. But that is a special case, and it is worth seeing what happens when we give
it up - because this notebook sits in a session on symplectic methods for a reason.

### Beam optics is Hamiltonian mechanics

Write the transverse motion as a Hamiltonian system with $s$ playing the role of
time:

$$H(x, p; s) = \underbrace{\frac{p^2}{2}}_{T(p)} + \underbrace{\frac{k(s)\,x^2}{2}}_{V(x;s)},
\qquad x' = \frac{\partial H}{\partial p} = p,
\qquad p' = -\frac{\partial H}{\partial x} = -k(s)\,x.$$

This is the harmonic oscillator of `pendulums.ipynb`, with a spring constant that
flips sign every couple of meters. It is **separable** - kinetic plus potential -
which is exactly the structure the drift-kick-drift splitting needs. Split it and
you get the leapfrog integrator; compose three weighted leapfrog substeps with
Yoshida's coefficients and you get the fourth-order method used in the other Session
20 notebooks, with no change beyond the force law.

The connection runs deeper than an analogy. A real $2\times2$ matrix is symplectic
if and only if its determinant is 1, so **every transfer matrix in this notebook is
already a symplectic map**. That single fact is why the emittance came out identical
at all three points in section 8: area preservation was built into the arithmetic
from the start.

### Who invented symplectic integrators, and why

Accelerator physicists did, for exactly this problem. A storage ring asks you to
follow a particle for $10^6$ to $10^9$ turns, and any integrator that leaks
phase-space area will report a beam that slowly damps or blows up - an artifact
indistinguishable from real physics. Ronald Ruth, at SLAC, published the first
explicit symplectic integrator in 1983 in the *IEEE Transactions on Nuclear Science*
under the title "A Canonical Integration Technique". Forest and Ruth extended it to
fourth order in 1990, and Yoshida published the general composition rule the same
year. The coefficients used in `henon_heiles.ipynb`, `pendulums.ipynb`, and
`planets.ipynb` come from that accelerator lineage.

Modern tracking codes still work this way. MAD-X and Elegant model a thick magnet as
drift - thin kick - drift, sliced as finely as needed, which is precisely the
leapfrog splitting. They accept a second-order integrator that is exactly symplectic
over a fourth-order one that is not.

### The experiment

We rebuild the one-cell map three ways - leapfrog, Yoshida fourth order, and
classical RK4 - and compare each against the exact matrix. Then we track a proton
for 4,000 turns with a deliberately coarse step and watch the Courant-Snyder
invariant,

$$J = \gamma x^2 + 2\alpha x x' + \beta x'^2,$$

which is the conserved quantity of betatron motion and the direct analogue of the
energy in `henon_heiles.ipynb`.

In [ ]:
# Cell 14 - Rebuild the cell map with three integrators and compare


def yoshida_coefficients() -> tuple[np.ndarray, np.ndarray]:
    """Position (c) and velocity (d) substep coefficients for Yoshida 4th order.

    These are the same coefficients used in henon_heiles.ipynb, pendulums.ipynb,
    and planets.ipynb. They depend only on the cube root of 2 and work for any
    separable Hamiltonian H = T(p) + V(q).
    """
    cbrt2 = 2.0 ** (1.0 / 3.0)
    w1 = 1.0 / (2.0 - cbrt2)
    w0 = -cbrt2 / (2.0 - cbrt2)
    c = np.array([w1 / 2.0, (w0 + w1) / 2.0, (w0 + w1) / 2.0, w1 / 2.0])
    d = np.array([w1, w0, w1])
    return c, d


C_SUBSTEP, D_SUBSTEP = yoshida_coefficients()


def step_leapfrog(state: np.ndarray, k: float, h: float) -> np.ndarray:
    """One drift-kick-drift step of length h. Second order, exactly symplectic."""
    x, p = state
    x = x + 0.5 * h * p  # drift
    p = p - h * k * x  # kick, evaluated at the updated position
    x = x + 0.5 * h * p  # drift
    return np.array([x, p])


def step_yoshida(state: np.ndarray, k: float, h: float) -> np.ndarray:
    """One Yoshida 4th-order step: three weighted leapfrog substeps."""
    x, p = state
    for j in range(3):
        x = x + C_SUBSTEP[j] * p * h
        p = p + D_SUBSTEP[j] * (-k * x) * h
    x = x + C_SUBSTEP[3] * p * h
    return np.array([x, p])


def step_rk4(state: np.ndarray, k: float, h: float) -> np.ndarray:
    """One classical Runge-Kutta 4 step. Fourth order, but not symplectic."""

    def derivative(u):
        return np.array([u[1], -k * u[0]])

    k1 = derivative(state)
    k2 = derivative(state + 0.5 * h * k1)
    k3 = derivative(state + 0.5 * h * k2)
    k4 = derivative(state + h * k3)
    return state + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def integrated_cell_map(stepper, max_slice: float) -> np.ndarray:
    """Build the horizontal one-cell map by integrating instead of using matrices.

    The motion is linear, so pushing the two basis vectors [1, 0] and [0, 1]
    through the whole cell recovers the 2x2 map that the integrator represents.
    """
    matrix = np.eye(2)
    for _name, _kind, length, kx, _ky in build_cell(K_FOCUS, K_DEFOCUS):
        n_slices = max(1, int(np.ceil(length / max_slice)))
        h = length / n_slices
        columns = []
        for basis in (np.array([1.0, 0.0]), np.array([0.0, 1.0])):
            state = basis
            for _ in range(n_slices):
                state = stepper(state, kx, h)
            columns.append(state)
        matrix = np.column_stack(columns) @ matrix
    return matrix


INTEGRATORS = [
    ("leapfrog", step_leapfrog, "seagreen"),
    ("Yoshida 4", step_yoshida, "royalblue"),
    ("RK4", step_rk4, "crimson"),
]
SLICE_LENGTHS = np.array([0.4, 0.2, 0.1, 0.05, 0.025, 0.0125])

map_error = {name: [] for name, _, _ in INTEGRATORS}
map_det_error = {name: [] for name, _, _ in INTEGRATORS}
for max_slice in SLICE_LENGTHS:
    for name, stepper, _color in INTEGRATORS:
        integrated = integrated_cell_map(stepper, max_slice)
        map_error[name].append(np.linalg.norm(integrated - matrix_x_cell))
        map_det_error[name].append(abs(np.linalg.det(integrated) - 1.0))

print("how far the integrated one-cell map is from the exact matrix")
print(f"{'slice (m)':>10}" + "".join(f"{name:>14}" for name, _, _ in INTEGRATORS))
for i, max_slice in enumerate(SLICE_LENGTHS):
    row = "".join(f"{map_error[name][i]:14.3e}" for name, _, _ in INTEGRATORS)
    print(f"{max_slice:10.4f}{row}")

print("\nhow far the determinant of that map is from 1")
print(f"{'slice (m)':>10}" + "".join(f"{name:>14}" for name, _, _ in INTEGRATORS))
for i, max_slice in enumerate(SLICE_LENGTHS):
    row = "".join(f"{map_det_error[name][i]:14.3e}" for name, _, _ in INTEGRATORS)
    print(f"{max_slice:10.4f}{row}")

print("\nRK4 is the most accurate per step and the only one that loses the")
print("determinant. Both symplectic maps hold it at 1 to machine precision,")
print("no matter how coarse the slicing gets.")

In [ ]:
# Cell 15 - Plot 9: accuracy per step versus survival over many turns

COARSE_SLICE = 0.25  # deliberately coarse, to make the drift visible
N_TURNS = 4000


def courant_snyder(state: np.ndarray) -> float:
    """The betatron invariant J, evaluated with the design Twiss parameters."""
    return (
        GAMMA_X0 * state[0] ** 2
        + 2.0 * ALPHA_X0 * state[0] * state[1]
        + BETA_X0 * state[1] ** 2
    )


def invariant_history(one_cell_map: np.ndarray, n_turns: int) -> np.ndarray:
    """Track one proton for n_turns and return the relative drift in J."""
    turn_map = np.linalg.matrix_power(one_cell_map, N_CELLS)
    state = np.array([0.005, 0.0])
    reference = courant_snyder(state)
    drift = np.empty(n_turns)
    for turn in range(n_turns):
        drift[turn] = abs(courant_snyder(state) / reference - 1.0)
        state = turn_map @ state
    return drift


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

ax = axes[0]
for name, _stepper, color in INTEGRATORS:
    ax.loglog(SLICE_LENGTHS, map_error[name], "o-", color=color, label=name)
ax.loglog(SLICE_LENGTHS, 0.7 * SLICE_LENGTHS**2, "k:", lw=1, label="slope 2")
ax.loglog(SLICE_LENGTHS, 0.6 * SLICE_LENGTHS**4, "k--", lw=1, label="slope 4")
ax.set_xticks(SLICE_LENGTHS)
ax.set_xticklabels([f"{length:g}" for length in SLICE_LENGTHS])
ax.xaxis.set_minor_formatter(NullFormatter())
ax.set_xlabel("slice length (m)")
ax.set_ylabel("error in the one-cell map")
ax.legend(fontsize=8)
ax.grid(True, which="both")
ax.set_title("Accuracy per step: RK4 wins")

ax = axes[1]
turns = np.arange(N_TURNS)
for name, stepper, color in INTEGRATORS[1:]:
    drift = invariant_history(integrated_cell_map(stepper, COARSE_SLICE), N_TURNS)
    ax.semilogy(turns, drift, color=color, lw=1.0, label=name)
exact_drift = invariant_history(matrix_x_cell, N_TURNS)
ax.semilogy(turns, exact_drift, color="gray", lw=1.0, label="exact matrix")
ax.set_xlabel("turn")
ax.set_ylabel(r"$|J / J_0 - 1|$")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, which="both")
ax.set_title(f"Survival over {N_TURNS:,} turns: symplectic wins")

fig.suptitle("Why Accelerator Physicists Invented Symplectic Integrators")
fig.tight_layout()
plt.show()

for name, stepper, _color in INTEGRATORS[1:]:
    drift = invariant_history(integrated_cell_map(stepper, COARSE_SLICE), N_TURNS)
    print(
        f"{name:10s} |dJ/J| after {N_TURNS:,} turns = {drift[-1]:.3e}, "
        f"largest seen = {drift.max():.3e}"
    )
print(
    f"{'exact':10s} |dJ/J| after {N_TURNS:,} turns = {exact_drift[-1]:.3e} "
    f"(pure roundoff)"
)

The two panels disagree about which integrator is better, and both are right.

On the left, RK4 is the most accurate method per step - roughly ten times better
than Yoshida at the same slice length, with both falling off as the fourth power
while leapfrog manages only the second. If the job were to cross one magnet once,
RK4 would be the obvious choice.

On the right that ordering inverts completely. RK4's one-cell map has a determinant
that differs from 1 by a few parts in $10^7$, so every turn multiplies the
phase-space area by a fixed factor slightly larger than 1 and the error compounds
**geometrically**: by turn 4,000 the invariant has grown by 2%, and it keeps going.
Yoshida's map has determinant 1 to machine precision, so there is no factor to
compound. Its error is a small bounded oscillation about the true value - the same
shadow-Hamiltonian behavior as the energy plot in `henon_heiles.ipynb` - and it
would still be a small bounded oscillation after a billion turns.

A 2% emittance growth looks exactly like real beam heating. It is not. It is the
integrator. That is why this class of methods exists, and why the accelerator
community built them before anyone else needed them.

---
## 11. What we left out

The model above is **linear** and **monochromatic**, which is enough for the core
ideas and not enough to build a machine. Real design codes such as MAD-X, Elegant,
and Accelerator Toolbox add:

| Effect | What it does | How it is handled |
|---|---|---|
| Momentum spread | Off-momentum particles are bent differently | Dispersion function $D(s)$, a third row in the matrices |
| Chromaticity | Focusing depends on momentum, so the tune smears | Sextupole magnets |
| Nonlinear fields | Sextupoles and errors drive resonances | Tracking millions of turns, dynamic aperture studies |
| Space charge | The beam repels itself | Self-consistent particle-in-cell simulation |
| Synchrotron radiation | Damps and excites the emittance | Radiation integrals, equilibrium emittance |
| Fringe fields | The field does not stop at the magnet end | Field maps, thick-element integrators |
| Acceleration | $B\rho$ changes, so $k$ must ramp with it | All magnets tracked against the momentum ramp |

Most of those additions are nonlinear, which is where section 10 stops being an
exercise and becomes mandatory: a sextupole has no closed-form transfer matrix, so
it must be integrated, and it must be integrated symplectically or a dynamic
aperture study will measure the integrator instead of the machine.

The framework, though, is the one Courant and Snyder wrote down in 1952, and it has
not changed. A modern lattice file is still a list of elements, a map per element, a
periodic Twiss solution, a beta function, and a working point chosen to dodge the
resonance lines.

### Summary

| Quantity | Symbol | What it tells you | Value here |
|---|---|---|---|
| Rigidity | $B\rho$ | How hard the beam is to bend | 10.01 T m |
| Focusing strength | $k$ | Quadrupole gradient normalized to the beam | 1.32 m$^{-2}$ |
| Phase advance | $\mu$ | Betatron phase gained per cell | 77 deg |
| Tune | $Q$ | Betatron oscillations per turn | 4.28, 4.19 |
| Beta function | $\beta(s)$ | How wide the beam is allowed to be | 1.9 to 7.8 m |
| Emittance | $\varepsilon$ | Beam quality, conserved | 5 mm mrad |
| Beam size | $\sigma$ | $\sqrt{\varepsilon\beta}$ | 3.1 to 6.2 mm |

### Related notebooks

`pendulums.ipynb`, `planets.ipynb`, and `henon_heiles.ipynb` apply the same Yoshida
coefficients used in section 10 to other separable Hamiltonians. This notebook is
where those coefficients came from historically, and the $|J/J_0 - 1|$ plot here is
the beam-optics version of the energy-drift plot in `henon_heiles.ipynb`.

### References

- Courant, E. D., Livingston, M. S. & Snyder, H. S. (1952). *The Strong-Focusing
  Synchrotron: A New High Energy Accelerator.* Physical Review, **88**, 1190-1196.
- Courant, E. D. & Snyder, H. S. (1958). *Theory of the Alternating-Gradient
  Synchrotron.* Annals of Physics, **3**, 1-48.
- Christofilos, N. C. (1950). *Focusing System for Ions and Electrons.* U.S. Patent
  2,736,799 (filed 1950, granted 1956).
- Courant, E. D. (2003). *Accelerators, Colliders, and Snakes.* Annual Review of
  Nuclear and Particle Science, **53**, 1-37.
- Ruth, R. D. (1983). *A Canonical Integration Technique.* IEEE Transactions on
  Nuclear Science, **NS-30**(4), 2669-2671. (The first explicit symplectic
  integrator, written for accelerator tracking.)
- Forest, E. & Ruth, R. D. (1990). *Fourth-Order Symplectic Integration.* Physica D,
  **43**(1), 105-117.
- Yoshida, H. (1990). *Construction of Higher Order Symplectic Integrators.* Physics
  Letters A, **150**(5-7), 262-268.
- Wiedemann, H. (2015). *Particle Accelerator Physics*, 4th ed. Springer. (Chapters
  7 and 8 cover everything in this notebook in full detail.)
- Wilson, E. J. N. (2001). *An Introduction to Particle Accelerators.* Oxford
  University Press.